# Fock-space tutorial: from occupations to the spectroscopy solver

This tutorial shows how a truncated Fock basis becomes a sector-based model that QuDPy-FDGF can propagate. The example uses two coupled bosonic modes and keeps the manifolds with total occupation $N=0,1,2,3$.

The important separation is:

1. **Physics layer:** choose the Fock states and construct $H_N$ and the rectangular ladder blocks.
2. **Adapter layer:** `ExcitationSectorModel` diagonalizes each manifold and exposes the common sector contract.
3. **Solver layer:** the dense and sparse backends propagate the same Liouville pathways without knowing how the basis states were generated.

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np
from qudpy_fdgf import (
    ExcitationSectorModel,
    FrequencyPathway,
    PropagationInterval,
    SpectroscopyProtocol,
    SpectroscopySolver,
    standard_nq_protocol,
)

## 1. A Fock basis is a list of occupations

For two bosonic modes, a basis state is the tuple $|n_a,n_b\rangle$. Grouping states by the conserved total occupation $N=n_a+n_b$ gives independent excitation manifolds.

For $M$ bosonic modes,

$$d_N=\binom{M+N-1}{N}, \qquad D_{\leq N_{\max}}=\binom{M+N_{\max}}{N_{\max}}.$$

The cutoff is therefore part of the physical approximation. It must include every manifold that the selected nonlinear pathways can reach. A third-order ESA pathway needs at least $N_{\max}=2$.

In [ ]:
def bosonic_sector_basis(n_modes, total_occupation):
    """Return all non-negative occupation tuples summing to N."""
    if n_modes == 1:
        return ((total_occupation,),)
    states = []
    for first in range(total_occupation, -1, -1):
        for tail in bosonic_sector_basis(n_modes - 1, total_occupation - first):
            states.append((first, *tail))
    return tuple(states)


n_modes = 2
Nmax = 3
bases = {N: bosonic_sector_basis(n_modes, N) for N in range(Nmax + 1)}

for N, basis in bases.items():
    expected = comb(n_modes + N - 1, N)
    assert len(basis) == expected
    print(f"N={N}: dimension={len(basis)}, basis={basis}")

D = sum(map(len, bases.values()))
assert D == comb(n_modes + Nmax, Nmax)
print(f"Total truncated Hilbert dimension: D={D}")

## 2. Construct number-conserving Hamiltonian blocks

We use

$$H=\omega_a n_a+\omega_b n_b+g(a^\dagger b+b^\dagger a)+\frac{U_a}{2}n_a(n_a-1)+\frac{U_b}{2}n_b(n_b-1)+U_{ab}n_an_b.$$

Because $[H,n_a+n_b]=0$, the Hamiltonian is block diagonal in $N$. The optical raising operator

$$J^+=\mu_a a^\dagger+\mu_b b^\dagger$$

connects adjacent manifolds. Its matrix elements contain the bosonic factors $\sqrt{n_i+1}$. This is the main concrete difference from hard-core spin excitations, where an occupied site cannot be raised again.

In [ ]:
def build_bosonic_hamiltonian(
    basis, omega_a, omega_b, coupling, U_a, U_b, U_ab
):
    index = {state: i for i, state in enumerate(basis)}
    H = np.zeros((len(basis), len(basis)), dtype=complex)

    for column, (n_a, n_b) in enumerate(basis):
        H[column, column] = (
            omega_a * n_a
            + omega_b * n_b
            + 0.5 * U_a * n_a * (n_a - 1)
            + 0.5 * U_b * n_b * (n_b - 1)
            + U_ab * n_a * n_b
        )
        if n_b:
            target = (n_a + 1, n_b - 1)
            H[index[target], column] += coupling * np.sqrt((n_a + 1) * n_b)
        if n_a:
            target = (n_a - 1, n_b + 1)
            H[index[target], column] += coupling * np.sqrt(n_a * (n_b + 1))

    assert np.allclose(H, H.conj().T)
    return H


def build_creation_block(source_basis, target_basis, mu_a, mu_b):
    """Matrix of J+ from one fixed-N manifold to the next."""
    target_index = {state: i for i, state in enumerate(target_basis)}
    block = np.zeros((len(target_basis), len(source_basis)), dtype=complex)
    for column, (n_a, n_b) in enumerate(source_basis):
        block[target_index[(n_a + 1, n_b)], column] += mu_a * np.sqrt(n_a + 1)
        block[target_index[(n_a, n_b + 1)], column] += mu_b * np.sqrt(n_b + 1)
    return block

In [ ]:
omega_a, omega_b = 1.50, 1.62
coupling = 0.025
U_a, U_b, U_ab = -0.040, -0.030, 0.020
mu_a, mu_b = 1.0, 0.60
eta = 0.015

hamiltonian_blocks = {
    N: build_bosonic_hamiltonian(
        basis, omega_a, omega_b, coupling, U_a, U_b, U_ab
    )
    for N, basis in bases.items()
}
raising_blocks = {
    (N + 1, N): build_creation_block(bases[N], bases[N + 1], mu_a, mu_b)
    for N in range(Nmax)
}

model = ExcitationSectorModel(
    hamiltonian_blocks,
    raising_blocks,
    initial_sector=0,
)

print("Raw block shapes")
for N in range(Nmax + 1):
    print(f"  H_{N}: {hamiltonian_blocks[N].shape}")
for key, block in raising_blocks.items():
    print(f"  J+_{key[1]}->{key[0]}: {block.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.7), constrained_layout=True)
images = [
    axes[0].imshow(np.real(hamiltonian_blocks[2]), cmap="RdBu_r"),
    axes[1].imshow(np.abs(raising_blocks[(2, 1)]), cmap="magma"),
]
axes[0].set_title(r"$\mathrm{Re}\,H_2$ in the Fock basis")
axes[1].set_title(r"$|J^+_{2\leftarrow1}|$")
for axis in axes:
    axis.set_xlabel("source-state index")
    axis.set_ylabel("target-state index")
for axis, image in zip(axes, images):
    fig.colorbar(image, ax=axis, shrink=0.82)

## 3. Translate the Fock construction into the common model contract

`ExcitationSectorModel` performs a basis change independently in every manifold:

$$H_N=V_NE_NV_N^\dagger, \qquad J^+_{N+1,N}\mapsto V_{N+1}^\dagger J^+_{N+1,N}V_N.$$

The rectangular transition blocks are therefore preserved even though neighboring sectors have different dimensions. From this point onward, the solver only asks for sector dimensions, Hamiltonian blocks, transition blocks, observables, and the initial condition. That interface is what is **fundamentally common** to this Fock model and the spin-chain model; their microscopic basis construction is not common.

In [ ]:
sector_dimensions = {N: model.dimension(N) for N in model.sectors()}
initial = model.initial_condition()

assert sector_dimensions == {N: len(bases[N]) for N in bases}
assert initial.sector == 0
assert model.transition_decomposition() == "explicit_sector"

for N in range(Nmax):
    plus = model.transition_blocks("light_matter", "plus", N)[N + 1]
    minus = model.transition_blocks("light_matter", "minus", N + 1)[N]
    assert plus.shape == (model.dimension(N + 1), model.dimension(N))
    assert np.allclose(minus, plus.conj().T)

print("Solver-facing sectors:", sector_dimensions)
print("Initial condition: sector", initial.sector, "state index 0")
print("Transition decomposition:", model.transition_decomposition())

## 4. What a Fock cutoff means inside a Liouville pathway

A density operator is organized into blocks $|N_{\mathrm{ket}}\rangle\langle N_{\mathrm{bra}}|$. Each interaction changes one side of this pair. The Fock labels are not discarded: they become the sector-pair labels used during pathway propagation.

For the ESA sequence `Bu -> Ku -> Ku`, the solver visits

$$ (0,0)\rightarrow(0,1)\rightarrow(1,1)\rightarrow(2,1). $$

The last interaction requires the two-quantum manifold. If the model stopped at $N_{\max}=1$, the pathway would be truncated for a physical reason rather than a numerical-backend reason.

In [ ]:
def trace_sector_pairs(sector_model, labels, initial_pair=(0, 0)):
    pairs = {initial_pair}
    history = [pairs]
    for label in labels:
        side = "ket" if label.startswith("K") else "bra"
        direction = "plus" if label in {"Ku", "Bu"} else "minus"
        updated = set()
        for ket_sector, bra_sector in pairs:
            source = ket_sector if side == "ket" else bra_sector
            targets = sector_model.transition_blocks(
                "light_matter", direction, source
            )
            for target in targets:
                pair = (target, bra_sector) if side == "ket" else (ket_sector, target)
                updated.add(pair)
        pairs = updated
        history.append(pairs)
    return history


esa_labels = ("Bu", "Ku", "Ku")
esa_history = trace_sector_pairs(model, esa_labels)
for step, pairs in enumerate(esa_history):
    action = "initial" if step == 0 else esa_labels[step - 1]
    print(f"{step}: after {action:>7s} -> {sorted(pairs)}")

assert esa_history[-1] == {(2, 1)}

## 5. The same Fock model can feed different solver backends

The dense backend is the small-system reference: it assembles explicit Liouville-space matrices. The sparse-sector backend keeps the sector structure and applies the generator without assembling the full dense superoperator. Both solve the same physical problem, so they should agree at small dimension.

In [ ]:
dense_solver = SpectroscopySolver(
    backend="dense_liouville", eta=eta, cache_resolvents=False
).feed_model(model)
sparse_solver = SpectroscopySolver(
    backend="sparse_sector", eta=eta, krylov_tolerance=1e-11
).feed_model(model)

print("Dense summary:", dense_solver.summary())
print("Sparse summary:", sparse_solver.summary())

In [ ]:
linear_pathway = FrequencyPathway(
    name="linear", interactions=("Ku",), component="linear"
)
linear_protocol = SpectroscopyProtocol(
    intervals=(PropagationInterval("omega", "frequency", coherence_order=1),),
    name="linear_frequency",
)
omega = np.linspace(1.40, 1.70, 61)

def linear_response(solver):
    return np.asarray([
        solver.calc_pathway(
            linear_pathway, linear_protocol, {"omega": frequency}
        ).value
        for frequency in omega
    ])

linear_dense = linear_response(dense_solver)
linear_sparse = linear_response(sparse_solver)
linear_residual = np.max(np.abs(linear_dense - linear_sparse))

assert np.allclose(linear_dense, linear_sparse, rtol=1e-8, atol=1e-9)
print(f"Dense--sparse linear residual: {linear_residual:.3e}")

fig, axes = plt.subplots(2, 1, figsize=(7.0, 5.4), sharex=True, constrained_layout=True)
axes[0].plot(omega, np.abs(linear_dense), linewidth=2.2, label="dense")
axes[0].plot(omega, np.abs(linear_sparse), "--", label="sparse sector")
axes[0].set_ylabel(r"$|P^{(1)}(\omega)|$")
axes[0].legend()
axes[1].semilogy(omega, np.maximum(np.abs(linear_dense - linear_sparse), 1e-18))
axes[1].set_xlabel(r"Energy $\omega$ (eV)")
axes[1].set_ylabel("absolute difference")

In [ ]:
esa_pathway = FrequencyPathway(
    name="ESA",
    interactions=esa_labels,
    component="rephasing",
)
esa_protocol = standard_nq_protocol(
    order=1,
    nq_interval=1,
    detection_interval=3,
    n_interactions=3,
    nq_axis="omega_1q",
    detection_axis="omega_emit",
)
esa_coordinates = {"omega_1q": -1.50, "t2": 0.0, "omega_emit": 1.48}

esa_dense = dense_solver.calc_pathway(
    esa_pathway, esa_protocol, esa_coordinates
).value
esa_sparse = sparse_solver.calc_pathway(
    esa_pathway, esa_protocol, esa_coordinates
).value

assert abs(esa_dense) > 1e-10
assert np.allclose(esa_dense, esa_sparse, rtol=1e-8, atol=1e-9)
print(f"ESA dense value:  {esa_dense:.8g}")
print(f"ESA sparse value: {esa_sparse:.8g}")
print(f"ESA residual:     {abs(esa_dense - esa_sparse):.3e}")

## 6. Where the Fock representation becomes expensive

The Hilbert dimension $D$ is only the beginning. A general density matrix contains $D^2$ complex amplitudes. An explicit dense Liouvillian contains $D^4$ complex entries, whereas a matrix-free sparse propagation still acts on vectors whose worst-case size scales as $D^2$.

Sector sparsity avoids storing impossible couplings and the full dense superoperator, but it does not by itself turn a high-rank many-body density matrix into a small object. The next cell shows the combinatorial growth before any solver operation is performed.

In [ ]:
def bosonic_truncated_dimension(n_modes, cutoff):
    return comb(n_modes + cutoff, cutoff)

def hardcore_truncated_dimension(n_sites, cutoff):
    return sum(comb(n_sites, N) for N in range(min(cutoff, n_sites) + 1))

mode_counts = np.arange(2, 13)
complex_bytes = np.dtype(np.complex128).itemsize

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), constrained_layout=True)
for cutoff in (1, 2, 3, 4):
    dimensions = np.asarray([
        bosonic_truncated_dimension(M, cutoff) for M in mode_counts
    ])
    axes[0].semilogy(mode_counts, dimensions, "o-", label=rf"$N_{{\max}}={cutoff}$")
axes[0].set_xlabel("number of bosonic modes $M$")
axes[0].set_ylabel("truncated Hilbert dimension $D$")
axes[0].set_title("Bosonic Fock-space growth")
axes[0].legend()

cutoff = 3
dimensions = np.asarray([bosonic_truncated_dimension(M, cutoff) for M in mode_counts])
dense_liouvillian_mib = complex_bytes * dimensions**4 / 2**20
sparse_vector_mib = complex_bytes * dimensions**2 / 2**20
axes[1].semilogy(mode_counts, dense_liouvillian_mib, "o-", label=r"dense matrix $D^4$")
axes[1].semilogy(mode_counts, sparse_vector_mib, "o-", label=r"state vector $D^2$")
axes[1].set_xlabel(r"number of modes $M$ at $N_{\max}=3$")
axes[1].set_ylabel("structural memory (MiB)")
axes[1].set_title("Objects seen by Liouville solvers")
axes[1].legend()

print("Example dimensions at Nmax=3")
for M in (2, 4, 8, 12):
    bosonic = bosonic_truncated_dimension(M, 3)
    hardcore = hardcore_truncated_dimension(M, 3)
    print(f"  M={M:2d}: bosonic D={bosonic:4d}, hard-core D={hardcore:4d}")

## 7. How well is the current solver adapted?

| Situation | Recommended interpretation |
|---|---|
| Small $D$, validation, debugging | `dense_liouville` is the clearest reference and exposes discrepancies. |
| Moderate $D$, sparse sector couplings | `sparse_sector` is the practical exact backend; it avoids the explicit $D^4$ Liouvillian. |
| Low excitation order and a natural $N_{\max}$ | The Fock-sector construction is especially well adapted because only reachable manifolds need to be retained. |
| Strong drive, high temperature, or large bosonic occupation | The required cutoff grows; convergence with $N_{\max}$ must be checked explicitly. |
| Large, high-rank dissipative systems | Sector sparsity may still leave a $D^2$ state-vector bottleneck. |
| Very large correlated chains | Tensor-network or other compressed-state methods need a different state contract; they are not obtained simply by changing the basis labels. |

The repository also contains an experimental low-rank Liouville backend. It can be useful when the density operator remains compressible, but it introduces a rank-truncation approximation and is not part of the stable default backend registry. It should therefore be benchmarked against the dense or sparse exact result before being trusted.

## 8. Takeaways

- A Fock space is a basis-construction choice, not a separate solver API.
- `ExcitationSectorModel` bridges fixed-occupation Hamiltonians and rectangular ladder blocks to the common `SectorModel` contract.
- Nonlinear pathways determine the minimum occupation cutoff: the ESA example reaches $N=2$.
- Dense and sparse backends should agree for the same truncated model; their difference is numerical representation, not physics.
- Sparse sectors delay the memory wall, but the general Liouville state can still scale as $D^2$.
- Always test convergence with respect to both the Fock cutoff and the numerical backend settings.